# Libraries

In [ ]:
import imageio.v3 as iio
import imageio
import matplotlib.pyplot as plt
import numpy as np
from skimage.feature import peak_local_max
from scipy.ndimage import gaussian_filter
from matplotlib.patches import Arc
import os
import cv2 
import niqe
import pandas as pd

# Directory Setup

In [ ]:
INPUT_FOLDER = r"C:\Users\makvanmt1\Downloads\test\test\Images\Gridded_Images"
OUTPUT_FOLDER = os.path.join(INPUT_FOLDER, "Star_Results")

if not os.path.exists(OUTPUT_FOLDER):
    os.makedirs(OUTPUT_FOLDER)

# Get list of all images
all_files = [f for f in os.listdir(INPUT_FOLDER) if f.lower().endswith(('.png', '.jpg', '.jpeg', '.tif'))]

if all_files:
    IMAGE_PATH = os.path.join(INPUT_FOLDER, all_files[0])
    try:
        data = iio.imread(IMAGE_PATH)
        if data.ndim == 3: 
            frame = np.mean(data, axis=-1).astype(np.uint8)
        else: 
            frame = data
        
        # Resize the initial frame to 256x256
        frame = cv2.resize(frame, (256, 256), interpolation=cv2.INTER_AREA)
        
        print(f"Loaded {len(all_files)} images. Previewing: {all_files[0]} (Resized to 256x256)")
    except Exception as e:
        print(f"Error loading images: {e}")
else:
    print("No images found in the specified directory.")

# Star Mask & Filtering

In [ ]:
def create_hex_star_mask(shape, cutoff_radius=60, rotation_deg=0):
    h, w = shape; cy, cx = h // 2, w // 2
    Y, X = np.ogrid[:h, :w] 
    fx, fy = X - cx, Y - cy
    rho, theta = np.sqrt(fx**2 + fy**2), np.arctan2(fy, fx)
    alpha_0 = np.radians(rotation_deg)
    l_ij = 0.75 + (np.cos(3 * (theta - alpha_0))**2) / 2
    f0 = cutoff_radius / np.max(l_ij)
    return (rho < (f0 * l_ij)).astype(float)

def histogram_stretch(image):
    img_min, img_max = image.min(), image.max()
    if img_max > img_min:
        return ((image - img_min) * (255.0 / (img_max - img_min))).astype(np.uint8)
    return np.zeros_like(image, dtype=np.uint8)

def apply_filter(magnitude_data, phase_data, mask):
    filtered_fft = mask * magnitude_data * np.exp(1j * phase_data)
    reconstructed = np.abs(np.fft.ifft2(np.fft.ifftshift(filtered_fft)))
    return histogram_stretch(reconstructed)

def find_mask_spikes(mask, cy, cx, num_spikes=6):
    h, w = mask.shape
    Y, X = np.indices((h, w))
    dy, dx = Y - cy, X - cx
    rho, theta = np.sqrt(dx**2 + dy**2), np.arctan2(dy, dx)
    bright = (mask > 0.5) & (rho > 10)
    if not np.any(bright): return [], []
    slice_w = np.pi / (num_spikes / 2)
    hist, bins = np.histogram(np.mod(theta[bright], slice_w), bins=100, range=(0, slice_w))
    dom_angle = ((bins[:-1] + bins[1:]) / 2.0)[np.argmax(hist)]
    mx, my = [], []
    for i in range(num_spikes):
        ang = dom_angle + i * slice_w
        spike = (np.cos(theta - ang) > 0.98) & bright
        if np.any(spike):
            idx = np.argmax(rho[spike])
            my.append(Y[spike][idx]); mx.append(X[spike][idx])
    return mx, my

# Frequency Domain Analysis and Angle Estimation

In [ ]:
fft_img = np.fft.fftshift(np.fft.fft2(frame))
magnitude, phase = np.abs(fft_img), np.angle(fft_img)
cy, cx = np.array(magnitude.shape) // 2
coords = peak_local_max(magnitude, min_distance=5, threshold_abs=0.01 * magnitude.max())
peaks = [np.degrees(np.arctan2(y-cy, x-cx)) % 360 for y, x in coords if np.sqrt((x-cx)**2 + (y-cy)**2) > 5]
theta_0 = np.mean(np.mod(peaks[:6], 60)) if peaks else 0.0

# Parameter Tuning

In [ ]:
def plot_interactive(cutoff_radius, rotation_deg, sigma):
    mask = gaussian_filter(create_hex_star_mask(frame.shape, cutoff_radius, rotation_deg), sigma=max(0.1, sigma))
    filtered = apply_filter(magnitude, phase, mask)
    mx, my = find_mask_spikes(mask, cy, cx)

    fig, axes = plt.subplot_mosaic([['orig', 'mask', 'filt'], ['orig', 'mask', 'hist']], figsize=(18, 7))
    axes['orig'].imshow(frame, cmap='gray'); axes['orig'].set_title('Original (256x256)'); axes['orig'].axis('off')
    axes['mask'].imshow(mask, cmap='hot'); axes['mask'].set_title('Spectral Mask'); axes['mask'].axis('off')
    for x, y in zip(mx, my): axes['mask'].plot([cx, x], [cy, y], 'c--', lw=0.8)
    axes['filt'].imshow(filtered, cmap='gray'); axes['filt'].set_title('Restored'); axes['filt'].axis('off')
    axes['hist'].hist(filtered.ravel(), bins=256, color='black', alpha=0.7); axes['hist'].set_title('Histogram')
    plt.tight_layout(); plt.show()

gui = interactive(plot_interactive, 
    cutoff_radius=widgets.IntSlider(value=60, min=10, max=128, description='Radius:'),
    rotation_deg=widgets.FloatSlider(value=theta_0, min=0.0, max=60.0, step=0.1, description='Rotation:'),
    sigma=widgets.IntSlider(value=4, min=0, max=20, description='Smooth:'))

In [ ]:
# Select the first image for a quick visual preview
if all_files:
    try:
        # 1. Load and preprocess a single image
        preview_path = os.path.join(INPUT_FOLDER, all_files[0])
        preview_data = iio.imread(preview_path)
        
        if preview_data.ndim == 3: 
            preview_frame = np.mean(preview_data, axis=-1).astype(np.uint8)
        else: 
            preview_frame = preview_data
        
        preview_frame = cv2.resize(preview_frame, (256, 256), interpolation=cv2.INTER_AREA)
        
        # 2. Apply FFT Analysis
        p_fft = np.fft.fftshift(np.fft.fft2(preview_frame))
        p_mag, p_phs = np.abs(p_fft), np.angle(p_fft)
        
        # 3. Generate mask and apply filter using your static parameters
        preview_mask = gaussian_filter(
            create_hex_star_mask(preview_frame.shape, CUTOFF_RADIUS, ROTATION_DEG), 
            sigma=max(0.1, SIGMA)
        )
        restored_preview = apply_filter(p_mag, p_phs, preview_mask)
        
        # 4. Calculate NIQE Scores
        score_in = niqe.niqe(preview_frame.astype(float))
        score_out = niqe.niqe(restored_preview.astype(float))
        
        # 5. Plot the comparison side-by-side
        fig, axes = plt.subplots(1, 2, figsize=(12, 6))
        
        axes[0].imshow(preview_frame, cmap='gray')
        axes[0].set_title(f"Original Preview\nNIQE: {score_in:.2f}")
        axes[0].axis('off')
        
        axes[1].imshow(restored_preview, cmap='gray')
        axes[1].set_title(f"Restored Preview\nNIQE: {score_out:.2f}")
        axes[1].axis('off')
        
        plt.tight_layout()
        plt.show()
        
    except Exception as e:
        print(f"Error generating preview: {e}")
else:
    print("No images found to preview.")

# Batch Processing, NIQE Evaluation, and CSV Export

In [ ]:
# Set static tuning parameters directly here instead of using sliders
CUTOFF_RADIUS = 60
ROTATION_DEG = theta_0
SIGMA = 4

niqe_records = []
print(f"Starting Batch Processing: {len(all_files)} images (Resizing to 256x256)...")

for filename in all_files:
    try:
        img_path = os.path.join(INPUT_FOLDER, filename)
        img_data = iio.imread(img_path)
        
        if img_data.ndim == 3: 
            proc_img = np.mean(img_data, axis=-1).astype(np.uint8)
        else: 
            proc_img = img_data
    
        proc_img = cv2.resize(proc_img, (256, 256), interpolation=cv2.INTER_AREA)
        
        score_orig = niqe.niqe(proc_img.astype(float))
        
        f_fft = np.fft.fftshift(np.fft.fft2(proc_img))
        f_mag, f_phs = np.abs(f_fft), np.angle(f_fft)
        
        mask = gaussian_filter(create_hex_star_mask(proc_img.shape, CUTOFF_RADIUS, ROTATION_DEG), sigma=max(0.1, SIGMA))
        final_img = apply_filter(f_mag, f_phs, mask)
        
        score_restored = niqe.niqe(final_img.astype(float))
        
        save_path = os.path.join(OUTPUT_FOLDER, f"Restored_{filename}")
        iio.imwrite(save_path, final_img)
        
        niqe_records.append({
            "Filename": filename,
            "Original_NIQE": round(score_orig, 2),
            "Restored_NIQE": round(score_restored, 2)
        })
        
        print(f"Processed: {filename} | Score: {score_restored:.2f}")
        
    except Exception as e:
        print(f"Error processing {filename}: {e}")

csv_path = os.path.join(OUTPUT_FOLDER, "NIQE_Scores_Report_256.csv")
df = pd.DataFrame(niqe_records)
df.to_csv(csv_path, index=False)

print(f"\n--- Batch Complete ---")
print(f"Images and CSV report saved in: {OUTPUT_FOLDER}")

# Preview

In [ ]:

if all_files:
    try:
        # 1. Load and preprocess a single image
        preview_path = os.path.join(INPUT_FOLDER, all_files[0])
        preview_data = iio.imread(preview_path)
        
        if preview_data.ndim == 3: 
            preview_frame = np.mean(preview_data, axis=-1).astype(np.uint8)
        else: 
            preview_frame = preview_data
        
        preview_frame = cv2.resize(preview_frame, (256, 256), interpolation=cv2.INTER_AREA)
        
        # 2. Apply FFT Analysis
        p_fft = np.fft.fftshift(np.fft.fft2(preview_frame))
        p_mag, p_phs = np.abs(p_fft), np.angle(p_fft)
        
        # 3. Generate mask and apply filter using your static parameters
        preview_mask = gaussian_filter(
            create_hex_star_mask(preview_frame.shape, CUTOFF_RADIUS, ROTATION_DEG), 
            sigma=max(0.1, SIGMA)
        )
        restored_preview = apply_filter(p_mag, p_phs, preview_mask)
        
        # 4. Calculate NIQE Scores
        score_in = niqe.niqe(preview_frame.astype(float))
        score_out = niqe.niqe(restored_preview.astype(float))
        
        # 5. Plot the comparison side-by-side
        fig, axes = plt.subplots(1, 2, figsize=(12, 6))
        
        axes[0].imshow(preview_frame, cmap='gray')
        axes[0].set_title(f"Original Preview\nNIQE: {score_in:.2f}")
        axes[0].axis('off')
        
        axes[1].imshow(restored_preview, cmap='gray')
        axes[1].set_title(f"Restored Preview\nNIQE: {score_out:.2f}")
        axes[1].axis('off')
        
        plt.tight_layout()
        plt.show()
        
    except Exception as e:
        print(f"Error generating preview: {e}")
else:
    print("No images found to preview.")